# Homework 6 - Trees

import Dataset:

In [1]:
#_____________________________________________________________________________________________________
#                                               imports                                               
#_____________________________________________________________________________________________________

import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

Dataset information:

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 859.5 KB


_____________________________________________________________________________________________________

Preparing the dataset:

In [3]:
#_____________________________________________________________________________________________________
#                                    Fill missing values with zeros                                   
#_____________________________________________________________________________________________________

df = df.fillna(0)


#_____________________________________________________________________________________________________
#                      Train | Validation | Test (exact calls from the homework)                      
#_____________________________________________________________________________________________________

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


#_____________________________________________________________________________________________________
#                                                Target                                               
#_____________________________________________________________________________________________________

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values


#_____________________________________________________________________________________________________
#                                 Encoder (DictVectorizer sparse=True)                                
#_____________________________________________________________________________________________________

train_dicts = df_train.drop(columns='fuel_efficiency_mpg').to_dict(orient='records')
val_dicts = df_val.drop(columns='fuel_efficiency_mpg').to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

feature_names = list(dv.get_feature_names_out())

print(len(df_train), len(df_val), len(df_test))

6000 2000 2000


_____________________________________________________________________________________________________

1. Decision tree with max_depth=1

In [4]:
#_____________________________________________________________________________________________________
#                                             Create model                                            
#_____________________________________________________________________________________________________

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)


#_____________________________________________________________________________________________________
#                                 Which feature is used for the split?                                
#_____________________________________________________________________________________________________

print(export_text(dt, feature_names=feature_names))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



_____________________________________________________________________________________________________

2. Random forest (n_estimators=10)

In [5]:
#_____________________________________________________________________________________________________
#                                             Create model                                            
#_____________________________________________________________________________________________________

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)


#_____________________________________________________________________________________________________
#                                           Predict and RMSE                                          
#_____________________________________________________________________________________________________

y_pred_val = rf.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_pred_val)

print(f'rmse_val: {rmse_val}')

rmse_val: 1.8332197358745623


_____________________________________________________________________________________________________

3. Best n_estimators

In [6]:
rmse_n_estimators = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred_val = rf.predict(X_val)
    rmse_n_estimators[n] = round(root_mean_squared_error(y_val, y_pred_val), 3)

    print(f'n_estimators={n:<4} rmse_val={rmse_n_estimators[n]}')

best_n = min(rmse_n_estimators, key=rmse_n_estimators.get)
print(f'best n_estimators: {best_n}')

n_estimators=10   rmse_val=1.833


n_estimators=50   rmse_val=1.774


n_estimators=100  rmse_val=1.77


n_estimators=150  rmse_val=1.769
best n_estimators: 150


_____________________________________________________________________________________________________

4. Best max_depth

In [7]:
mean_rmse_depth = {}

for d in [10, 15, 20, 25]:
    scores = []

    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred_val = rf.predict(X_val)
        scores.append(root_mean_squared_error(y_val, y_pred_val))

    mean_rmse_depth[d] = np.mean(scores)
    print(f'max_depth={d:<3} mean_rmse={round(mean_rmse_depth[d], 4)}')

best_depth = min(mean_rmse_depth, key=mean_rmse_depth.get)
print(f'best max_depth: {best_depth}')

max_depth=10  mean_rmse=1.7544


max_depth=15  mean_rmse=1.7842


max_depth=20  mean_rmse=1.7868


max_depth=25  mean_rmse=1.7863
best max_depth: 10


_____________________________________________________________________________________________________

5. Feature importance

In [8]:
#_____________________________________________________________________________________________________
#                                             Create model                                            
#_____________________________________________________________________________________________________

rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)


#_____________________________________________________________________________________________________
#                                          Feature importance                                         
#_____________________________________________________________________________________________________

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print(importances.head(8))


#_____________________________________________________________________________________________________
#                                 Among the 4 options of the question                                 
#_____________________________________________________________________________________________________

options = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print(importances[options].sort_values(ascending=False))

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
dtype: float64
vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64


_____________________________________________________________________________________________________

6. XGBoost (eta)

In [9]:
#_____________________________________________________________________________________________________
#                                        DMatrix and watchlist                                        
#_____________________________________________________________________________________________________

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)

watchlist = [(dtrain, 'train'), (dval, 'val')]


#_____________________________________________________________________________________________________
#                                      Train with eta 0.3 and 0.1                                     
#_____________________________________________________________________________________________________

rmse_eta = {}

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)

    y_pred_val = model.predict(dval)
    rmse_eta[eta] = root_mean_squared_error(y_val, y_pred_val)

    print(f'eta={eta} rmse_val={round(rmse_eta[eta], 4)}')

eta=0.3 rmse_val=1.8265


eta=0.1 rmse_val=1.7248
